# Manual2AR：手动运行的 SAM3.1 + Cutie GPU 服务

1. 在 Google Colab 中上传本笔记本（仅这一份文件），选择 T4 GPU 运行时。
2. 默认参数即可，点击「全部运行」。无需 Colab CLI、私有仓库授权或本地 API Key。
3. 等待安装、预加载及 HTTPS 隧道启动，浏览器会下载 `inference.connection.local.json`。如果下载被阻止，从左侧文件浏览器的 `/content/Manual2ARLearningPerformanceV2/` 下载同名文件。
4. 将文件放到本地项目 `configs/inference.connection.local.json`，覆盖旧连接；浏览器追加的 `(1)` 等后缀要移除。重启本地后端，在 Unity 中重新识别，旧跟踪会话不能继续使用。
5. 使用期间保留 Colab 页面。实验结束后通过 Colab 菜单断开并删除运行时；停止 Python 服务不等于释放 GPU。

默认不挂载 Google Drive、不创建云盘缓存、不持续写入云盘日志；首次启动需要联网安装依赖和下载模型，等待可能较久。若要复用已有缓存，可勾选 `USE_DRIVE_CACHE`；若需要持续留存远端日志，可勾选 `PERSIST_REMOTE_LOGS`，两者均会请求 Drive 授权。

连接文件含访问令牌，请仅保存在本地 configs 中，不上传 Git、不发送到聊天。千问 API Key 仍保留在本地项目配置中，不上传至 Colab。

### 可选 Drive 缓存

`DRIVE_CACHE_ROOT` 默认沿用 `/content/drive/MyDrive/Manual2AR/cache-v1`。启用后校验并复制模型、锁定的 wheel 安装包到 Colab 本地磁盘，在本地安装和执行；不直接从 Drive 执行 venv。

首次建立环境缓存可能占用数 GB 至十几 GB 云盘空间。`CACHE_ENVIRONMENT=False` 仅缓存模型与配置；`FORCE_REBUILD_ENV=True` 仅在修复环境时使用，成功后恢复 False。

参数优先级：原始默认配置 → 已启用的 Drive 配置缓存 → 本次表单。令牌及临时服务地址不进入缓存；每次新运行时都需下载新的连接文件。现场图像不自动上传 Drive。

缓存只减少安装、下载等待，不保证零网络、秒开或避免 Colab 回收。手动启动也不承诺固定会话时长。


# Manual2AR 共享视觉：SAM3.1 + Cutie 持续会话服务

选择 NVIDIA GPU 运行时，然后 Run all。Notebook 自动准备独立 Python3.12、SAM3.1 FP16首帧初始化、Cutie Mega；T4默认使用Cutie FP16，可切换FP32做对照。无需私有仓库、千问Key或Hugging Face授权。本服务与旧跟踪对比服务协议不同，支持一个会话内的多目标持续记忆：首帧SAM3.1分割，之后每个新帧仅运行Cutie。模型在服务进程内常驻，会话记忆相互独立，闲置5分钟释放。

服务源码已内嵌。下载新连接文件放到本地 `configs/inference.connection.local.json`，在共享视觉选择 SAM3.1 + Cutie。只有本地主动识别、跟踪时才上传图像；GPU日志默认记录指纹、面积、耗时和错误，不保存原图。静态图片也可识别分割，但不模拟运动。

Runpod可填已有Pod的HTTP8021 HTTPS代理地址；本笔记本不申请或停止收费资源。

In [ ]:
PUBLIC_GPU_URL = ""  # @param {type:"string"}
CUTIE_PRECISION = "auto"  # @param ["auto", "float32", "float16", "bfloat16"]
CUTIE_SHORT_SIDE = -1  # @param [-1, 320, 256]
CUTIE_MEM_EVERY = 5  # @param [5, 10]
CHANNELS_LAST = False  # @param {type:"boolean"}
COMPILE_ENCODER = False  # @param {type:"boolean"}
RETAIN_INITIAL_IMAGES = False  # @param {type:"boolean"}
# Colab留空自动创建临时隧道；Runpod填 https://<POD_ID>-8021.proxy.runpod.net

# Drive 缓存；首次需要在Google弹窗授权挂载。模型/性能选项继续使用上面的表单。
USE_DRIVE_CACHE = False  # @param {type:"boolean"}
DRIVE_CACHE_ROOT = "/content/drive/MyDrive/Manual2AR/cache-v1"  # @param {type:"string"}
CACHE_ENVIRONMENT = True  # @param {type:"boolean"}
FORCE_REBUILD_ENV = False  # @param {type:"boolean"}
PERSIST_REMOTE_LOGS = False  # @param {type:"boolean"}
REMOTE_LOG_INTERVAL = 15  # @param {type:"integer"}


In [ ]:
import base64,io,json,os,pathlib,sys,zipfile
BASE=pathlib.Path('/content/Manual2ARLearningPerformanceV2') if pathlib.Path('/content').exists() else pathlib.Path.cwd()/'Manual2ARLearningPerformanceV2'
BASE.mkdir(parents=True,exist_ok=True)
SOURCE_ZIP="..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP))) as archive:archive.extractall(BASE)
required=['manual2ar_inference/engines/sam31_sources.json','configs/inference.example.json','deploy/common/requirements.txt','manual2ar_inference/engines/sam31_image.py','deploy/common/weights.py']
missing=[name for name in required if not (BASE/name).is_file()]
if missing:raise RuntimeError('Notebook源码包缺少文件，请重新下载新版笔记本：'+', '.join(missing))
os.chdir(BASE)
config_path=BASE/'configs/inference.local.json'
if not config_path.exists():
    config_path.write_bytes((BASE/'configs/inference.example.json').read_bytes())
config_path.chmod(0o600)
if USE_DRIVE_CACHE or PERSIST_REMOTE_LOGS:
    try:
        from google.colab import drive
    except ImportError:
        raise RuntimeError('Drive留存需要Colab；其他平台请取消USE_DRIVE_CACHE和PERSIST_REMOTE_LOGS。')
    drive.mount('/content/drive')
    cache_path=pathlib.Path(DRIVE_CACHE_ROOT).expanduser().resolve() if USE_DRIVE_CACHE else None
    if cache_path is not None and not cache_path.is_relative_to(pathlib.Path('/content/drive').resolve()):
        raise ValueError('DRIVE_CACHE_ROOT必须位于已挂载的/content/drive内。')
else:
    cache_path=None
from deploy.colab.drive_cache import DriveCache
drive_cache=DriveCache(BASE,cache_path,cache_environment=CACHE_ENVIRONMENT)
config=drive_cache.settings(json.loads(config_path.read_text()))
config['precision']=CUTIE_PRECISION
config['cutie'].update(max_internal_size=int(CUTIE_SHORT_SIDE),mem_every=int(CUTIE_MEM_EVERY),channels_last=CHANNELS_LAST,compile_encoder=COMPILE_ENCODER)
config['retain_initial_images']=RETAIN_INITIAL_IMAGES
config_path.write_text(json.dumps(config,indent=2))
print('源码与性能参数已准备；模型固定SAM3.1 + Cutie。')

print('Drive配置恢复后，由本次表单覆盖性能选项；API token不写入Drive。')

from deploy.colab.retention import start_recorder
if not 5<=REMOTE_LOG_INTERVAL<=300:raise ValueError('日志间隔需在5至300秒')
retention_path=start_recorder(BASE,{'enabled':PERSIST_REMOTE_LOGS,'path':'/content/drive/MyDrive/Manual2AR/logs','interval_seconds':REMOTE_LOG_INTERVAL})
print('远端日志留存目录：',retention_path)


In [ ]:
import shutil
from deploy.colab.environment import prepare_environment,run_logged,child_environment
(BASE/'bootstrap.log').write_text('Notebook Python: '+sys.version+'\n')
try:
    if not shutil.which('nvidia-smi'):raise RuntimeError('请先选择NVIDIA GPU运行时。')
    GPU_PYTHON=drive_cache.prepare(force=FORCE_REBUILD_ENV)
except Exception:
    print('安装失败，实际错误及最后输出见上方；完整日志在 bootstrap.log。')
    try:
        from google.colab import files
        files.download(str(BASE/'bootstrap.log'))
    except Exception:print('可从左侧文件浏览器下载 bootstrap.log。')
    raise


In [ ]:
import subprocess,time,urllib.request
if 'service' in globals() and service is not None and service.poll() is None:
    service.terminate();service.wait(timeout=15)
config=json.loads(config_path.read_text())
service_log=open(BASE/'gpu_service.log','w')
service=subprocess.Popen([GPU_PYTHON,'-u','-m','manual2ar_inference.app','--config',str(config_path),'--work-dir',str(BASE)],env=child_environment(),stdout=service_log,stderr=subprocess.STDOUT)
from deploy.colab.worker import read,private_json,process_identity
processes=read(BASE/'worker-processes.local.json',{})
processes['service']=process_identity(service.pid)
private_json(BASE/'worker-processes.local.json',processes)
health=None
for _ in range(120):
    if service.poll() is not None:raise RuntimeError('服务退出，请查看 gpu_service.log。')
    try:
        request=urllib.request.Request('http://127.0.0.1:'+str(config['port'])+'/health',headers={'Authorization':'Bearer '+config['api_token']})
        with urllib.request.urlopen(request,timeout=2) as response:health=json.load(response)
        break
    except Exception:time.sleep(.5)
if health is None:raise RuntimeError('服务未就绪，请查看 gpu_service.log。')
if not health.get('available'):raise RuntimeError('GPU模型不可用：'+str(health.get('reason')))
print(json.dumps(health,ensure_ascii=False,indent=2))
warm_request=urllib.request.Request('http://127.0.0.1:'+str(config['port'])+'/v1/warmup',data=b'{}',headers={'Authorization':'Bearer '+config['api_token'],'Content-Type':'application/json'})
with urllib.request.urlopen(warm_request,timeout=300) as response:warm=json.load(response)
print('Cutie权重已预加载；首次SAM3.1初始化、CUDA内核和可选编译仍有冷启动。',round(warm['model_load_ms']),'ms')

# Persist small torchvision/ResNet downloads made during Cutie warmup.
drive_cache.save_auxiliary()


In [ ]:
import re
if 'tunnel' in globals() and tunnel is not None and tunnel.poll() is None:tunnel.terminate()
tunnel=None
if PUBLIC_GPU_URL:
    public_url=PUBLIC_GPU_URL.rstrip('/')
else:
    binary=BASE/'cloudflared'
    if not binary.exists():
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',binary)
        binary.chmod(0o700)
    tunnel_log_path=BASE/'tunnel.log';tunnel_log=open(tunnel_log_path,'w')
    tunnel=subprocess.Popen([str(binary),'tunnel','--url','http://127.0.0.1:'+str(config['port']),'--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
    public_url=None
    for _ in range(120):
        match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',tunnel_log_path.read_text(errors='replace'))
        if match:public_url=match.group(0);break
        if tunnel.poll() is not None:break
        time.sleep(.5)
    if not public_url:raise RuntimeError('临时隧道失败，请查看 tunnel.log。')
connection=BASE/'inference.connection.local.json'
connection.write_text(json.dumps(dict(url=public_url,api_token=config['api_token'],protocol='manual2ar-learning-cutie-v2'),indent=2));connection.chmod(0o600)
print('服务地址：',public_url)
print('将连接文件放在本地 configs/inference.connection.local.json；不要提交Git或公开token。')
try:
    from google.colab import files
    files.download(str(connection))
except ImportError:print('请从Jupyter文件浏览器下载 inference.connection.local.json。')

from deploy.colab.worker import read,private_json,process_identity
processes=read(BASE/'worker-processes.local.json',{})
processes['tunnel']=process_identity(tunnel.pid) if tunnel is not None else None
private_json(BASE/'worker-processes.local.json',processes)


## 使用与停止

本地：`bash run.sh start`，Unity打开Welcome场景，选择CPU或SAM3.1+Cutie，启动摄像头/加载静态图，再点击识别。重新识别、清除、切换输入或引擎会关闭旧会话。云端异常不回退CPU；请查看本次诊断日志。

首次模型加载会比较慢。当前使用无积压的顺序会话，Unity实时预览独立刷新；标签图单独更新，超过配置年龄即隐藏。configs/app.local.json的performance.preview_mode可选live或paired；paired只用于检查同帧对齐。云端延迟仍限制结果更新率，尚未承诺端到端实时性能。

默认保持原始尺寸、每5帧更新记忆。480不会缩小640×360录像；320/256、channels_last、编码器编译、记忆间隔10须逐项验证质量。更改参数后Run all下载新连接文件。GPU日志包含H2D、CUDA计时、D2H、显存、图像解码、标签图编码；本地性能报告加入往返及Unity回执。

停止下方服务后，Colab还需断开并删除运行时；Runpod还需在平台停止/终止Pod，关闭服务不等于停止资源计费。

In [ ]:
STOP_SERVICE=False  # @param {type:"boolean"}
if STOP_SERVICE:
    service.terminate()
    if tunnel is not None:tunnel.terminate()
    print('已请求停止服务；请在平台另行确认GPU资源状态。')

if STOP_SERVICE:
    from deploy.colab.worker import read,stop_owned
    stop_owned(read(BASE/'worker-processes.local.json',{}).get('recorder'))
